# 05 — EDA + IndoBERT vs ExtraTreesRegressor
## SIPEDULI — Perbandingan Model Klasikal vs Transformer

**Tujuan notebook ini:**
1. **EDA** — analisis mendalam dataset gabungan (real scraping + synthetic user reports)
2. **Fine-tune IndoBERT** — untuk regresi risk_score, dilatih pada dataset yang sama
3. **Perbandingan** — ExtraTrees vs IndoBERT: metrik, kecepatan, kemampuan konteks
4. **Kesimpulan deployment** — kenapa ExtraTrees lebih cocok untuk Railway

> **Catatan:** Fine-tuning IndoBERT membutuhkan GPU. Jalankan di Google Colab jika tidak punya GPU lokal.
> Di Colab: Runtime → Change runtime type → GPU (T4)

```
Dataset: 3.013 baris (2.413 real scraping + 600 synthetic user reports)
Model A: ExtraTreesRegressor + TF-IDF (notebook 04) → production
Model B: IndoBERT-base-p1 + regression head → perbandingan akademis
```

In [ ]:
# CELL DIAGNOSTIK — jalankan ini PERTAMA, lihat outputnya sebelum lanjut
import sys
print("Python exe :", sys.executable)
print("Python ver :", sys.version)

import numpy as np
print("numpy ver  :", np.__version__)
print("numpy path :", np.__file__)

import scipy
print("scipy ver  :", scipy.__version__)

import torch
print("torch ver  :", torch.__version__)
print("CUDA       :", torch.cuda.is_available())

# Jika bukan venv311, ganti kernel dulu!
if 'venv311' not in sys.executable:
    print()
    print("!!! KERNEL SALAH — bukan venv311 !!!")
    print("Ganti kernel: klik nama kernel di pojok kanan atas →")
    print("'Python 3.11 (venv311 SIPEDULI)'")
else:
    print()
    print("Kernel OK: venv311")

## 0. Setup & Install

In [ ]:
import os, sys, time, json, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from pathlib import Path

warnings.filterwarnings('ignore')

try:
    import google.colab
    IN_COLAB = True
    print('Running in Google Colab')
    ROOT = Path('/content/drive/MyDrive/sipeduli')
except ImportError:
    IN_COLAB = False
    print('Running locally')
    NOTEBOOK_DIR = Path().resolve()
    ROOT = NOTEBOOK_DIR
    while not (ROOT / 'data').exists() and ROOT != ROOT.parent:
        ROOT = ROOT.parent

DATA_PATH    = ROOT / 'data' / 'dataset_siap_training.csv'
MODEL_DIR    = ROOT / 'backend' / 'ml' / 'models'
DOCS_DIR     = ROOT / 'docs'
INDOBERT_DIR = ROOT / 'backend' / 'ml' / 'models' / 'indobert'
INDOBERT_DIR.mkdir(parents=True, exist_ok=True)
DOCS_DIR.mkdir(parents=True, exist_ok=True)

import torch
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device  : {DEVICE}')
if torch.cuda.is_available():
    print(f'GPU     : {torch.cuda.get_device_name(0)}')
else:
    print('PERINGATAN: CPU only — training IndoBERT akan sangat lambat (~10+ jam)')

import transformers
print(f'transformers: {transformers.__version__}')

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
torch.manual_seed(RANDOM_STATE)

print(f'\nROOT      : {ROOT}')
print(f'DATA_PATH : {DATA_PATH} | exists: {DATA_PATH.exists()}')

In [ ]:
import sys, torch
print("Python :", sys.executable)
print("Torch  :", torch.__version__)
print("CUDA   :", torch.cuda.is_available())
print("CUDA ver:", torch.version.cuda)

# Diagnosa lebih dalam kalau CUDA False
if not torch.cuda.is_available():
    print()
    print("=== DIAGNOSA ===")
    try:
        torch.cuda.init()
    except Exception as e:
        print("CUDA init error:", e)
    
    import subprocess
    r = subprocess.run([sys.executable, "-c",
        "import torch; print(torch.cuda.is_available())"],
        capture_output=True, text=True)
    print("Subprocess CUDA:", r.stdout.strip())
    print("Stderr:", r.stderr[:200] if r.stderr else "none")

---
# BAGIAN 1 — EDA (Exploratory Data Analysis)

EDA ini berfokus pada tiga pertanyaan:
1. Seperti apa distribusi dataset gabungan kita?
2. Apa perbedaan karakteristik teks berita (real) vs laporan user (synthetic)?
3. Kenapa IndoBERT dibutuhkan — kasus apa yang TF-IDF tidak bisa tangani?

## 1. Load Dataset & Overview

In [ ]:
import json
import scipy.sparse as sp
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans

df_full  = pd.read_csv(DATA_PATH)
df_real  = df_full[df_full['is_synthetic'] == 'No'].copy().reset_index(drop=True)
df_user  = df_full[df_full['is_synthetic'] == 'User'].copy().reset_index(drop=True)
df_smote = df_full[df_full['is_synthetic'] == 'Yes'].copy()
n_real   = len(df_real)

print('=' * 55)
print('  OVERVIEW DATASET')
print('=' * 55)
print(f'  Real scraping (No)     : {len(df_real):,} baris')
print(f'  Synthetic user (User)  : {len(df_user):,} baris')
print(f'  SMOTE lama (Yes)       : {len(df_smote):,} baris (tidak dipakai)')
print(f'  TOTAL TRAINING         : {len(df_real)+len(df_user):,} baris')
print()
print('Kolom dataset:')
print(df_full.dtypes.to_string())

# ── Load urgency signals dari model metadata ──────────────────
meta            = json.load(open(MODEL_DIR / 'model_metadata.json'))
URGENCY_SIGNALS = meta['urgency_signals']
FEATURE_COLS    = list(URGENCY_SIGNALS.keys())

def hitung_urgency(teks):
    words = str(teks).lower().split()
    n = max(len(words), 1)
    return {f: sum(words.count(kw) for kw in URGENCY_SIGNALS[f]) / n for f in FEATURE_COLS}

# ── KMeans clustering pada SEMUA data (tanpa label_urgensi) ───
print('\nMembangun risk_score dari KMeans clustering...')
bersih_all = pd.concat(
    [df_real['deskripsi_bersih'].fillna(''),
     df_user['deskripsi_bersih'].fillna('')],
    ignore_index=True
)
urg_all  = pd.DataFrame([hitung_urgency(t) for t in bersih_all])
scaler_  = StandardScaler()
X_scaled = scaler_.fit_transform(urg_all[FEATURE_COLS].values)

URGENCY_WEIGHTS = {
    'skor_kematian': 3.0, 'skor_senjata': 2.0, 'skor_pidana_berat': 2.0,
    'skor_kekerasan_fisik': 1.5, 'skor_perampasan_paksa': 1.0,
    'skor_harta': 1.0, 'skor_ringan': 0.5,
}
km       = KMeans(n_clusters=3, n_init=30, random_state=RANDOM_STATE)
km.fit(X_scaled)
w_arr    = np.array([URGENCY_WEIGHTS.get(f, 1.0) for f in FEATURE_COLS])
crit_idx = int(np.argmax(km.cluster_centers_ @ w_arr))
crit_ctr = km.cluster_centers_[crit_idx]

dists    = np.linalg.norm(X_scaled - crit_ctr, axis=1)
d_min, d_max = dists.min(), dists.max()
risk_all = 100.0 * (1.0 - (dists - d_min) / (d_max - d_min + 1e-9))

# Simpan risk_score dan kategori ke df_real / df_user
df_real['risk_score'] = risk_all[:n_real]
df_user['risk_score'] = risk_all[n_real:]
for df in [df_real, df_user]:
    df['kategori'] = pd.cut(
        df['risk_score'], bins=[-1, 34, 67, 101],
        labels=['Rendah', 'Sedang', 'Tinggi']
    ).astype(str)

print(f'Critical centroid: cluster {crit_idx}')
print(f'Risk score: min={risk_all.min():.1f}  max={risk_all.max():.1f}  mean={risk_all.mean():.1f}')
print('\nDistribusi kategori dari clustering (tanpa label_urgensi):')
all_kat = pd.concat([df_real['kategori'], df_user['kategori']])
print(all_kat.value_counts().reindex(['Tinggi', 'Sedang', 'Rendah']).to_string())

## 2. Distribusi Risk Score

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

df_all = pd.concat([df_real, df_user], ignore_index=True)

# ── Plot 1: Distribusi risk_score dari clustering ─────────────
for kat, color in [('Tinggi','#cc0000'), ('Sedang','#ffc107'), ('Rendah','#28a745')]:
    subset = df_all[df_all['kategori'] == kat]['risk_score']
    axes[0].hist(subset, bins=25, color=color, alpha=0.75, label=kat)
axes[0].axvline(67, color='black', linestyle='--', alpha=0.5, label='Thr Tinggi (67)')
axes[0].axvline(34, color='gray',  linestyle='--', alpha=0.5, label='Thr Sedang (34)')
axes[0].set_title('Distribusi Risk Score\nDari KMeans Clustering (semua data)', fontweight='bold')
axes[0].set_xlabel('Risk Score')
axes[0].set_ylabel('Frekuensi')
axes[0].legend(fontsize=8)
axes[0].grid(alpha=0.3)

# ── Plot 2: Distribusi kategori clustering per sumber ─────────
label_order = ['Tinggi', 'Sedang', 'Rendah']
real_counts = df_real['kategori'].value_counts().reindex(label_order, fill_value=0)
user_counts = df_user['kategori'].value_counts().reindex(label_order, fill_value=0)

x = np.arange(len(label_order))
w = 0.35
bars1 = axes[1].bar(x - w/2, real_counts.values, w, label='Real scraping', color='#0a0a0a', alpha=0.7)
bars2 = axes[1].bar(x + w/2, user_counts.values, w, label='Synthetic user', color='#3498db', alpha=0.7)
axes[1].set_xticks(x)
axes[1].set_xticklabels(label_order)
axes[1].set_title('Distribusi Kategori dari Clustering\nReal vs Synthetic', fontweight='bold')
axes[1].set_ylabel('Jumlah')
axes[1].legend()
axes[1].grid(alpha=0.3, axis='y')
for bar in bars1:
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 10,
                 str(int(bar.get_height())), ha='center', fontsize=8)
for bar in bars2:
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 10,
                 str(int(bar.get_height())), ha='center', fontsize=8)

# ── Plot 3: Komposisi dataset ─────────────────────────────────
sizes  = [len(df_real), len(df_user)]
labels = [f'Real scraping\n({len(df_real):,} baris)', f'Synthetic user\n({len(df_user):,} baris)']
axes[2].pie(sizes, labels=labels, colors=['#0a0a0a', '#3498db'],
            autopct='%1.1f%%', startangle=90, textprops={'fontsize': 9})
axes[2].set_title('Komposisi\nDataset Training', fontweight='bold')

plt.suptitle('Analisis Dataset Gabungan SIPEDULI', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(DOCS_DIR / '05a_dataset_overview.png', dpi=150, bbox_inches='tight')
plt.show()
print('Plot disimpan!')

## 3. Analisis Panjang Teks — Real vs Synthetic

In [ ]:
df_real['word_count'] = df_real['deskripsi_bersih'].fillna('').apply(lambda x: len(str(x).split()))
df_user['word_count'] = df_user['deskripsi_bersih'].fillna('').apply(lambda x: len(str(x).split()))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# ── Histogram word count ──────────────────────────────────────
axes[0].hist(df_real['word_count'], bins=50, color='#0a0a0a', alpha=0.7,
             label=f'Real scraping (mean={df_real["word_count"].mean():.0f})')
axes[0].hist(df_user['word_count'], bins=30, color='#3498db', alpha=0.7,
             label=f'Synthetic user (mean={df_user["word_count"].mean():.0f})')
axes[0].set_xlabel('Jumlah Kata (setelah stemming)')
axes[0].set_ylabel('Frekuensi')
axes[0].set_title('Distribusi Panjang Teks\nReal vs Synthetic', fontweight='bold')
axes[0].legend()
axes[0].grid(alpha=0.3)
axes[0].set_xlim(0, 250)

# ── Boxplot word count per kategori (dari clustering) ─────────
bp_data, bp_labels, bp_colors = [], [], []
for kat in ['Tinggi', 'Sedang', 'Rendah']:
    for src, df, color in [('Real', df_real, '#cc0000'), ('Synthetic', df_user, '#3498db')]:
        sub = df[df['kategori'] == kat]['word_count'].dropna()
        bp_data.append(sub.values)
        bp_labels.append(f'{kat}\n({src})')
        bp_colors.append(color)

bp = axes[1].boxplot(bp_data, patch_artist=True, labels=bp_labels)
for patch, color in zip(bp['boxes'], bp_colors):
    patch.set_facecolor(color)
    patch.set_alpha(0.6)
axes[1].set_ylabel('Jumlah Kata')
axes[1].set_title('Word Count per Kategori Clustering\nReal vs Synthetic', fontweight='bold')
axes[1].tick_params(axis='x', labelsize=7)
axes[1].grid(alpha=0.3, axis='y')

plt.suptitle('Analisis Panjang Teks — Distribution Shift', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(DOCS_DIR / '05b_text_length_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

print('Statistik word count:')
print(f'  Real scraping  — mean: {df_real["word_count"].mean():.1f}  '
      f'median: {df_real["word_count"].median():.1f}  max: {df_real["word_count"].max()}')
print(f'  Synthetic user — mean: {df_user["word_count"].mean():.1f}  '
      f'median: {df_user["word_count"].median():.1f}  max: {df_user["word_count"].max()}')

## 4. Distribusi Tipe Kejahatan

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Tipe kejahatan dari data real (kategori_kejahatan)
real_tipe = df_real['kategori_kejahatan'].value_counts().head(12)
user_tipe = df_user['kategori_kejahatan'].value_counts()

axes[0].barh(real_tipe.index[::-1], real_tipe.values[::-1], color='#0a0a0a', alpha=0.8)
axes[0].set_title('Tipe Kejahatan — Data Real Scraping', fontweight='bold')
axes[0].set_xlabel('Jumlah Berita')
axes[0].grid(alpha=0.3, axis='x')
for i, v in enumerate(real_tipe.values[::-1]):
    axes[0].text(v + 2, i, str(v), va='center', fontsize=8)

colors_tipe = ['#cc0000' if i < len(user_tipe)//3 else
               '#ffc107' if i < 2*len(user_tipe)//3 else '#28a745'
               for i in range(len(user_tipe))]
axes[1].barh(user_tipe.index[::-1], user_tipe.values[::-1],
             color=colors_tipe[::-1], alpha=0.8)
axes[1].set_title('Tipe Kejahatan — Synthetic User Reports', fontweight='bold')
axes[1].set_xlabel('Jumlah Laporan')
axes[1].grid(alpha=0.3, axis='x')
for i, v in enumerate(user_tipe.values[::-1]):
    axes[1].text(v + 0.5, i, str(v), va='center', fontsize=8)

plt.suptitle('Distribusi Tipe Kejahatan', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(DOCS_DIR / '05c_crime_type_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. Heatmap Urgency Features per Kategori

In [ ]:
# URGENCY_SIGNALS, FEATURE_COLS, hitung_urgency sudah didefinisikan di Section 1

# Hitung urgency features untuk seluruh training data
print('Menghitung urgency features untuk seluruh training data...')
df_train_all = pd.concat([df_real, df_user], ignore_index=True)
urg_rows = [hitung_urgency(t) for t in df_train_all['deskripsi_bersih'].fillna('')]
df_urg = pd.DataFrame(urg_rows)
df_urg['kategori'] = df_train_all['kategori'].values   # dari KMeans clustering
df_urg['sumber']   = df_train_all['is_synthetic'].values

# ── Heatmap mean urgency per kategori clustering ──────────────
fig, axes = plt.subplots(1, 2, figsize=(15, 4))

for ax, sumber, title_suffix in [
    (axes[0], 'No',   'Data Real Scraping'),
    (axes[1], 'User', 'Synthetic User Reports'),
]:
    subset = df_urg[df_urg['sumber'] == sumber]
    pivot  = subset.groupby('kategori')[FEATURE_COLS].mean().reindex(['Tinggi', 'Sedang', 'Rendah'])
    pivot.columns = [c.replace('skor_', '').replace('_', ' ').title() for c in pivot.columns]

    sns.heatmap(pivot, annot=True, fmt='.4f', cmap='RdYlGn',
                linewidths=0.5, ax=ax, cbar_kws={'label': 'Mean Score'})
    ax.set_title(f'Mean Urgency Features per Kategori Clustering\n{title_suffix}', fontweight='bold')
    ax.set_ylabel('Kategori (dari KMeans)')

plt.suptitle('Urgency Signal Features — Real vs Synthetic', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(DOCS_DIR / '05d_urgency_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()
print('Plot disimpan!')

## 6. Mengapa IndoBERT? — Kasus Context Disambiguation

TF-IDF memperlakukan setiap kata secara independen — tidak memahami konteks kalimat.
Ini menyebabkan dua kategori masalah yang tidak bisa diselesaikan model klasikal:

In [ ]:
import joblib, re
from Sastrawi.Stemmer.StemmerFactory import StemmerFactory
from scipy.sparse import hstack

extratrees = joblib.load(MODEL_DIR / 'model_final.pkl')
vectorizer  = joblib.load(MODEL_DIR / 'vectorizer.pkl')
stemmer     = StemmerFactory().create_stemmer()
THR_TINGGI, THR_SEDANG = 67.0, 34.0

def bersihkan(teks):
    teks = str(teks).lower()
    teks = re.sub(r'[^a-z\s]', ' ', teks)
    return stemmer.stem(teks)

def predict_extratrees_raw(teks):
    """Prediksi ExtraTrees TANPA post-processing rules — murni model."""
    clean = bersihkan(teks)
    X_tfidf = vectorizer.transform([clean])
    words = clean.split()
    n = max(len(words), 1)
    X_num = sp.csr_matrix([[sum(words.count(kw) for kw in URGENCY_SIGNALS[f]) / n
                            for f in FEATURE_COLS]])
    X = hstack([X_tfidf, X_num])
    skor = float(np.clip(extratrees.predict(X)[0], 0, 100))
    kat  = 'Tinggi' if skor >= THR_TINGGI else 'Sedang' if skor >= THR_SEDANG else 'Rendah'
    return skor, kat

# Kasus yang membuktikan keterbatasan TF-IDF
cases = [
    ('SAFE_CONTEXT — kata berbahaya dalam konteks aman',
     'Rendah',
     [
         'tadi beli celurit di pasar untuk berkebun di sawah',
         'anak saya main pistol mainan hadiah ulang tahun kemarin',
         'saya jual koleksi pisau dapur bekas di marketplace',
     ]),
    ('ACTIVE_SIGNALS — frasa darurat yang diremehkan model',
     'Tinggi',
     [
         'tolong segera datang pak ada korban luka sekarang',
         'ada orang tidak sadarkan diri di jalan mohon segera kirim bantuan',
         'kondisi kritis darurat butuh bantuan segera',
     ]),
]

print('=' * 70)
print('  KETERBATASAN TF-IDF — Contoh Kasus Nyata')
print('=' * 70)
for kategori_masalah, target, texts in cases:
    print(f'\nKasus: {kategori_masalah}')
    print(f'Target yang benar: {target}')
    print('-' * 70)
    for t in texts:
        skor, kat = predict_extratrees_raw(t)
        masalah = 'SALAH (butuh rule)' if kat != target else 'OK'
        print(f'  [{masalah}] {skor:.1f} → {kat}')
        print(f'   "{t}"')
print()
print('IndoBERT memahami konteks kalimat secara holistik →')
print('"beli celurit di pasar" berbeda dengan "orang bawa celurit mengancam"')

---
# BAGIAN 2 — Fine-tuning IndoBERT

**Model:** `indobenchmark/indobert-base-p1`  
**Task:** Regresi — prediksi `risk_score` (float 0–100)  
**Input:** Teks laporan ASLI (bukan stemmed) — IndoBERT punya tokenizer sendiri  
**Dataset:** 3.013 baris (sama dengan ExtraTrees)

> **Metodologi:** Label `risk_score` dihasilkan dari **KMeans clustering yang identik dengan notebook 04**,
> bukan dari kolom `label_urgensi` (yang berasal dari keyword scraping).
> Ini memastikan kedua model (ExtraTrees dan IndoBERT) belajar dari target yang **sama persis**,
> sehingga perbandingan bersifat apple-to-apple.

## 7. Persiapan Data untuk IndoBERT — KMeans Labeling (Tanpa label_urgensi)

In [ ]:
from sklearn.model_selection import train_test_split
from torch.utils.data import Dataset, DataLoader
from transformers import BertTokenizer

INDOBERT_MODEL_NAME = 'indobenchmark/indobert-base-p1'
MAX_LENGTH = 256
BATCH_SIZE = 8 if str(DEVICE) == 'cuda' else 4

teks_real = df_real['deskripsi_original'].fillna('').reset_index(drop=True)
teks_user = df_user['deskripsi_kejadian'].fillna('').reset_index(drop=True)

df_bert = pd.DataFrame({
    'teks':       pd.concat([teks_real, teks_user], ignore_index=True).values,
    'risk_score': np.concatenate([df_real['risk_score'].values,
                                  df_user['risk_score'].values]),
    'kategori':   pd.concat([df_real['kategori'], df_user['kategori']],
                            ignore_index=True).values,
})
df_bert = df_bert[df_bert['teks'].str.len() > 10].reset_index(drop=True)

# Min-max normalisasi agar target pakai seluruh range [0,1]
# (bukan /100 yang hanya memakai range [0.67, 1.0])
SCORE_MIN = float(df_bert['risk_score'].min())
SCORE_MAX = float(df_bert['risk_score'].max())

print(f'Dataset IndoBERT (label dari KMeans, tanpa label_urgensi):')
print(f'  Total baris : {len(df_bert):,}')
print(f'  Risk score  : min={SCORE_MIN:.1f}  max={SCORE_MAX:.1f}  mean={df_bert["risk_score"].mean():.1f}')
print(f'  Normalisasi : min-max → [0, 1]  (denorm saat inferensi)')
print('\nDistribusi kategori dari clustering:')
kat_counts = df_bert['kategori'].value_counts().reindex(['Tinggi', 'Sedang', 'Rendah'])
print(kat_counts.to_string())

min_count = kat_counts.min()
if min_count >= 2:
    train_df, test_df = train_test_split(
        df_bert, test_size=0.2, random_state=RANDOM_STATE,
        stratify=df_bert['kategori']
    )
    print(f'\nStratified split (min kategori: {min_count})')
else:
    train_df, test_df = train_test_split(
        df_bert, test_size=0.2, random_state=RANDOM_STATE
    )
    print(f'\nRandom split (ada kategori dengan hanya {min_count} sampel)')

print(f'Train: {len(train_df):,} | Test: {len(test_df):,}')
print(f'\nLoading tokenizer {INDOBERT_MODEL_NAME}...')
tokenizer = BertTokenizer.from_pretrained(INDOBERT_MODEL_NAME)
print(f'Tokenizer loaded! Vocab size: {tokenizer.vocab_size:,}')

In [ ]:
class LaporanDataset(Dataset):
    def __init__(self, texts, scores, tokenizer, max_length, score_min, score_max):
        self.encodings = tokenizer(
            texts.tolist(),
            max_length=max_length,
            padding='max_length',
            truncation=True,
            return_tensors='pt',
        )
        # Min-max normalize ke [0.001, 0.999] — strict range untuk BCEWithLogitsLoss
        normalized = (scores.values - score_min) / (score_max - score_min + 1e-9)
        normalized = np.clip(normalized, 0.001, 0.999)
        self.scores = torch.tensor(normalized, dtype=torch.float32)

    def __len__(self):
        return len(self.scores)

    def __getitem__(self, idx):
        return {
            'input_ids':      self.encodings['input_ids'][idx],
            'attention_mask': self.encodings['attention_mask'][idx],
            'labels':         self.scores[idx],
        }

print('Menyiapkan dataset...')
train_dataset = LaporanDataset(train_df['teks'], train_df['risk_score'],
                               tokenizer, MAX_LENGTH, SCORE_MIN, SCORE_MAX)
test_dataset  = LaporanDataset(test_df['teks'],  test_df['risk_score'],
                               tokenizer, MAX_LENGTH, SCORE_MIN, SCORE_MAX)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,  num_workers=0)
test_loader  = DataLoader(test_dataset,  batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

print(f'Train batches : {len(train_loader)}')
print(f'Test batches  : {len(test_loader)}')
print(f'Target range  : [0.001, 0.999]  (min-max normalized dari [{SCORE_MIN:.1f}, {SCORE_MAX:.1f}])')
sample = next(iter(train_loader))
print(f'Label sample  : min={sample["labels"].min():.3f}  max={sample["labels"].max():.3f}')

## 8. Definisi Model IndoBERT + Regression Head

In [ ]:
from transformers import BertModel
import torch.nn as nn

class IndoBERTRegressor(nn.Module):
    """IndoBERT-base + mean pooling + 2-layer head.
    Output: raw logits (BCEWithLogitsLoss saat training, sigmoid saat inferensi)."""
    def __init__(self, model_name: str, dropout: float = 0.2, freeze_layers: int = 2):
        super().__init__()
        self.bert      = BertModel.from_pretrained(model_name)
        self.dropout   = nn.Dropout(dropout)
        self.regressor = nn.Sequential(
            nn.Linear(768, 256),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(256, 1),
        )
        # Freeze hanya embedding + 2 layer pertama — beri kapasitas lebih besar
        modules_to_freeze = [self.bert.embeddings] + \
                            list(self.bert.encoder.layer[:freeze_layers])
        for module in modules_to_freeze:
            for param in module.parameters():
                param.requires_grad = False

    def mean_pooling(self, token_embeddings, attention_mask):
        mask = attention_mask.unsqueeze(-1).expand(token_embeddings.size()).float()
        return torch.sum(token_embeddings * mask, 1) / \
               torch.clamp(mask.sum(1), min=1e-9)

    def forward(self, input_ids, attention_mask):
        outputs = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        pooled  = self.mean_pooling(outputs.last_hidden_state, attention_mask)
        # Raw logits — BCEWithLogitsLoss lebih numerically stable dari BCE(sigmoid(x))
        return self.regressor(self.dropout(pooled)).squeeze(-1)

print(f'Loading IndoBERT dari {INDOBERT_MODEL_NAME}...')
indobert_model = IndoBERTRegressor(INDOBERT_MODEL_NAME, dropout=0.2, freeze_layers=2).to(DEVICE)

total_params     = sum(p.numel() for p in indobert_model.parameters())
trainable_params = sum(p.numel() for p in indobert_model.parameters() if p.requires_grad)
frozen_params    = total_params - trainable_params
print(f'Model loaded ke       : {next(indobert_model.parameters()).device}')
print(f'Total parameters      : {total_params:,}')
print(f'Trainable parameters  : {trainable_params:,}  ({trainable_params/total_params*100:.1f}%)')
print(f'Frozen parameters     : {frozen_params:,}  (embedding + 2 layer pertama)')
if torch.cuda.is_available():
    print(f'VRAM terpakai: {torch.cuda.memory_allocated()/1e9:.2f} GB / '
          f'{torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')

## 9. Training Loop

In [ ]:
from transformers import get_cosine_schedule_with_warmup
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from torch.cuda.amp import GradScaler, autocast

EPOCHS       = 7
GRAD_ACCUM   = 4
LR_BERT      = 2e-5
LR_HEAD      = 1e-3
WARMUP_RATIO = 0.1
PATIENCE     = 3
USE_AMP      = str(DEVICE) == 'cuda'

bert_params = [p for n, p in indobert_model.named_parameters()
               if p.requires_grad and 'regressor' not in n]
head_params = [p for n, p in indobert_model.named_parameters()
               if p.requires_grad and 'regressor' in n]

optimizer = torch.optim.AdamW([
    {'params': bert_params, 'lr': LR_BERT},
    {'params': head_params, 'lr': LR_HEAD},
], weight_decay=0.01)

# BCEWithLogitsLoss: dirancang untuk logit + target [0,1]
# Gradient: sigmoid(logit) - target → selalu aktif, tidak vanish
loss_fn     = nn.BCEWithLogitsLoss()
grad_scaler = GradScaler(enabled=USE_AMP)

total_steps = (len(train_loader) // GRAD_ACCUM) * EPOCHS
scheduler   = get_cosine_schedule_with_warmup(
    optimizer,
    num_warmup_steps=int(total_steps * WARMUP_RATIO),
    num_training_steps=total_steps,
)

history = {'train_loss': [], 'val_rmse': [], 'val_mae': [], 'val_r2': []}
SCORE_RANGE = SCORE_MAX - SCORE_MIN

print(f'Mulai training IndoBERT (BCE + min-max norm) pada {DEVICE}')
print(f'  Epochs             : {EPOCHS}  (early stopping patience={PATIENCE})')
print(f'  Effective batch    : {BATCH_SIZE * GRAD_ACCUM}')
print(f'  LR BERT / Head     : {LR_BERT} / {LR_HEAD}')
print(f'  Loss               : BCEWithLogitsLoss')
print(f'  Target scale       : min-max [{SCORE_MIN:.1f}, {SCORE_MAX:.1f}] → [0,1]')
print(f'  Mixed precision    : {USE_AMP}')
if USE_AMP:
    print(f'  VRAM               : {torch.cuda.memory_allocated()/1e9:.2f} GB / '
          f'{torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')
print()

best_val_rmse  = float('inf')
no_improve_cnt = 0
start_total    = time.time()

for epoch in range(EPOCHS):
    indobert_model.train()
    epoch_loss  = 0.0
    start_epoch = time.time()
    optimizer.zero_grad()

    for step, batch in enumerate(train_loader):
        input_ids      = batch['input_ids'].to(DEVICE)
        attention_mask = batch['attention_mask'].to(DEVICE)
        labels         = batch['labels'].to(DEVICE)   # [0,1] min-max normalized

        with autocast(enabled=USE_AMP):
            logits = indobert_model(input_ids, attention_mask)
            loss   = loss_fn(logits, labels) / GRAD_ACCUM

        grad_scaler.scale(loss).backward()
        epoch_loss += loss.item() * GRAD_ACCUM

        if (step + 1) % GRAD_ACCUM == 0:
            grad_scaler.unscale_(optimizer)
            nn.utils.clip_grad_norm_(indobert_model.parameters(), 1.0)
            grad_scaler.step(optimizer)
            grad_scaler.update()
            scheduler.step()
            optimizer.zero_grad()

        if (step + 1) % 50 == 0:
            elapsed   = time.time() - start_epoch
            vram_info = f'{torch.cuda.memory_allocated()/1e9:.2f}GB' if USE_AMP else 'N/A'
            print(f'  Epoch {epoch+1}/{EPOCHS} | Step {step+1}/{len(train_loader)} '
                  f'| Loss: {loss.item()*GRAD_ACCUM:.5f} | {elapsed:.0f}s | VRAM: {vram_info}')

    avg_loss = epoch_loss / len(train_loader)
    history['train_loss'].append(avg_loss)

    # ── Validasi — denormalize ke skala asli [0,100] untuk metrik ──
    indobert_model.eval()
    all_preds, all_labels = [], []
    with torch.no_grad():
        for batch in test_loader:
            with autocast(enabled=USE_AMP):
                logits = indobert_model(batch['input_ids'].to(DEVICE),
                                        batch['attention_mask'].to(DEVICE))
            # Denormalize: sigmoid(logit) → [0,1] → [SCORE_MIN, SCORE_MAX]
            preds_norm = torch.sigmoid(logits).cpu().float().numpy()
            preds_orig = preds_norm * SCORE_RANGE + SCORE_MIN
            labels_orig = batch['labels'].numpy() * SCORE_RANGE + SCORE_MIN
            all_preds.extend(preds_orig)
            all_labels.extend(labels_orig)

    all_preds = np.clip(all_preds, 0, 100)
    val_rmse  = float(np.sqrt(mean_squared_error(all_labels, all_preds)))
    val_mae   = float(mean_absolute_error(all_labels, all_preds))
    val_r2    = float(r2_score(all_labels, all_preds))

    history['val_rmse'].append(val_rmse)
    history['val_mae'].append(val_mae)
    history['val_r2'].append(val_r2)

    epoch_time = time.time() - start_epoch
    print(f'\nEpoch {epoch+1}/{EPOCHS} selesai — {epoch_time:.0f}s')
    print(f'  Train Loss  : {avg_loss:.5f}  (BCE)')
    print(f'  Val RMSE    : {val_rmse:.4f}')
    print(f'  Val MAE     : {val_mae:.4f}')
    print(f'  Val R²      : {val_r2:.4f}')
    if USE_AMP:
        print(f'  VRAM peak   : {torch.cuda.max_memory_allocated()/1e9:.2f} GB')
        torch.cuda.reset_peak_memory_stats()

    if val_rmse < best_val_rmse:
        best_val_rmse  = val_rmse
        no_improve_cnt = 0
        indobert_model.bert.save_pretrained(INDOBERT_DIR)
        tokenizer.save_pretrained(INDOBERT_DIR)
        torch.save(indobert_model.state_dict(), INDOBERT_DIR / 'regressor_head.pt')
        # Simpan juga normalization constants untuk inferensi
        json.dump({'score_min': SCORE_MIN, 'score_max': SCORE_MAX},
                  open(INDOBERT_DIR / 'norm_params.json', 'w'))
        print(f'  ✓ Model terbaik disimpan (RMSE: {best_val_rmse:.4f})')
    else:
        no_improve_cnt += 1
        print(f'  Tidak ada peningkatan ({no_improve_cnt}/{PATIENCE})')
        if no_improve_cnt >= PATIENCE:
            print(f'\nEarly stopping di epoch {epoch+1}')
            break
    print()

total_time    = time.time() - start_total
INDOBERT_RMSE = best_val_rmse
INDOBERT_MAE  = val_mae
INDOBERT_R2   = val_r2

print('=' * 55)
print(f'Training selesai  : {total_time/60:.1f} menit')
print(f'Best Val RMSE     : {best_val_rmse:.4f}')
print(f'Best Val R²       : {val_r2:.4f}')
if USE_AMP:
    print(f'GPU               : {torch.cuda.get_device_name(0)}')

## 10. Learning Curve IndoBERT

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

epochs_x = range(1, len(history['train_loss']) + 1)

# ── Training loss ─────────────────────────────────────────────
axes[0].plot(epochs_x, history['train_loss'], 'o-', color='#cc0000', linewidth=2, label='Train Loss (Huber)')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Huber Loss')
axes[0].set_title('Training Loss IndoBERT', fontweight='bold')
axes[0].legend()
axes[0].grid(alpha=0.3)

# ── Val RMSE + R² ─────────────────────────────────────────────
ax2 = axes[1].twinx()
axes[1].plot(epochs_x, history['val_rmse'], 'o-', color='#cc0000', linewidth=2, label='Val RMSE')
ax2.plot(epochs_x,     history['val_r2'],   's--', color='#28a745', linewidth=2, label='Val R²')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('RMSE', color='#cc0000')
ax2.set_ylabel('R²', color='#28a745')
axes[1].set_title('Validation RMSE & R²', fontweight='bold')
axes[1].legend(loc='upper left')
ax2.legend(loc='upper right')
axes[1].grid(alpha=0.3)

# ── Val MAE ───────────────────────────────────────────────────
axes[2].plot(epochs_x, history['val_mae'], 'o-', color='#3498db', linewidth=2, label='Val MAE')
axes[2].set_xlabel('Epoch')
axes[2].set_ylabel('MAE')
axes[2].set_title('Validation MAE', fontweight='bold')
axes[2].legend()
axes[2].grid(alpha=0.3)
# Annotate best epoch
best_epoch = int(np.argmin(history['val_rmse'])) + 1
axes[2].axvline(best_epoch, color='gray', linestyle='--', alpha=0.6, label=f'Best epoch={best_epoch}')

plt.suptitle('IndoBERT Training Curve (Mean Pooling + HuberLoss)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(DOCS_DIR / '05e_indobert_training_curve.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'Best epoch: {best_epoch}  |  RMSE: {min(history["val_rmse"]):.4f}  |  R²: {max(history["val_r2"]):.4f}')

---
# BAGIAN 3 — Perbandingan Model

## 11. Tabel Perbandingan Metrik

In [ ]:
# Metrik ExtraTrees (dari notebook 04)
ET_RMSE = meta.get('rmse_test', 4.21)   # ambil dari metadata jika tersimpan
ET_MAE  = meta.get('mae_test_score', 2.13)
ET_R2   = meta.get('r2_test_score', 0.9579)

# Inferensi speed test
test_teks = "pak polisi ada perampokan bersenjata tolong segera datang ada korban luka"
test_clean = bersihkan(test_teks)

# ExtraTrees speed
N_ITER = 100
t0 = time.time()
for _ in range(N_ITER):
    X_tfidf = vectorizer.transform([test_clean])
    words   = test_clean.split()
    n       = max(len(words), 1)
    X_num   = sp.csr_matrix([[sum(words.count(kw) for kw in URGENCY_SIGNALS[f]) / n
                               for f in FEATURE_COLS]])
    _ = extratrees.predict(hstack([X_tfidf, X_num]))
ET_SPEED = (time.time() - t0) / N_ITER * 1000  # ms

# IndoBERT speed
indobert_model.eval()
enc = tokenizer(test_teks, max_length=MAX_LENGTH, padding='max_length',
                truncation=True, return_tensors='pt')
with torch.no_grad():
    t0 = time.time()
    for _ in range(10):  # lebih sedikit karena lambat
        _ = indobert_model(enc['input_ids'].to(DEVICE), enc['attention_mask'].to(DEVICE))
    BERT_SPEED = (time.time() - t0) / 10 * 1000  # ms

# Ukuran model
import os
ET_SIZE_MB   = (os.path.getsize(MODEL_DIR/'model_final.pkl') +
                os.path.getsize(MODEL_DIR/'vectorizer.pkl')) / 1e6
BERT_SIZE_MB = sum(os.path.getsize(f) for f in INDOBERT_DIR.rglob('*') if f.is_file()) / 1e6

# Tabel perbandingan
comparison = pd.DataFrame({
    'Metrik': ['R² (Test)', 'RMSE (Test)', 'MAE (Test)',
               'Inferensi (ms/request)', 'Ukuran model', 'RAM saat load',
               'Training time', 'SHAP support', 'Deployment Railway'],
    'ExtraTreesRegressor': [
        f'{ET_R2:.4f}', f'{ET_RMSE:.4f}', f'{ET_MAE:.4f}',
        f'{ET_SPEED:.2f} ms', f'{ET_SIZE_MB:.0f} MB', '~200 MB',
        '~20 menit (CPU)', 'Ya (TreeExplainer)', 'Bisa (512MB free tier)'
    ],
    'IndoBERT': [
        f'{INDOBERT_R2:.4f}', f'{INDOBERT_RMSE:.4f}', f'{INDOBERT_MAE:.4f}',
        f'{BERT_SPEED:.0f} ms', f'{BERT_SIZE_MB:.0f} MB', '~2 GB',
        f'~{total_time/60:.0f} menit (GPU)', 'Tidak langsung*', 'Tidak (OOM)'
    ]
})

print('=' * 75)
print('  PERBANDINGAN MODEL')
print('=' * 75)
print(comparison.to_string(index=False))
print()
print('* IndoBERT explainability: bisa pakai attention visualization atau LIME')
print('  (tidak compatible dengan SHAP TreeExplainer)')

## 12. Visualisasi Perbandingan Metrik

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 5))

models  = ['ExtraTrees', 'IndoBERT']
colors  = ['#0a0a0a', '#3498db']

# ── R² ────────────────────────────────────────────────────────
r2_vals = [ET_R2, INDOBERT_R2]
bars = axes[0].bar(models, r2_vals, color=colors, alpha=0.85, edgecolor='black', width=0.5)
y_min_r2 = max(0, min(r2_vals) - 0.15)
axes[0].set_ylim(y_min_r2, 1.05)
axes[0].set_title('R² Score\n(lebih tinggi = lebih baik)', fontweight='bold')
axes[0].set_ylabel('R²')
axes[0].grid(alpha=0.3, axis='y')
for bar, val in zip(bars, r2_vals):
    axes[0].text(bar.get_x() + bar.get_width()/2,
                 bar.get_height() + 0.01,
                 f'{val:.4f}', ha='center', va='bottom', fontweight='bold', fontsize=10)

# ── RMSE ──────────────────────────────────────────────────────
rmse_vals = [ET_RMSE, INDOBERT_RMSE]
bars = axes[1].bar(models, rmse_vals, color=colors, alpha=0.85, edgecolor='black', width=0.5)
axes[1].set_ylim(0, max(rmse_vals) * 1.25)
axes[1].set_title('RMSE\n(lebih rendah = lebih baik)', fontweight='bold')
axes[1].set_ylabel('RMSE')
axes[1].grid(alpha=0.3, axis='y')
for bar, val in zip(bars, rmse_vals):
    axes[1].text(bar.get_x() + bar.get_width()/2,
                 bar.get_height() + max(rmse_vals) * 0.02,
                 f'{val:.4f}', ha='center', va='bottom', fontweight='bold', fontsize=10)

# ── Kecepatan Inferensi (linear scale) ───────────────────────
speed_vals = [ET_SPEED, BERT_SPEED]
bars = axes[2].bar(models, speed_vals, color=colors, alpha=0.85, edgecolor='black', width=0.5)
axes[2].set_ylim(0, max(speed_vals) * 1.3)
axes[2].set_title('Kecepatan Inferensi (ms)\n(lebih rendah = lebih baik)', fontweight='bold')
axes[2].set_ylabel('ms / request')
axes[2].grid(alpha=0.3, axis='y')
for bar, val in zip(bars, speed_vals):
    axes[2].text(bar.get_x() + bar.get_width()/2,
                 bar.get_height() + max(speed_vals) * 0.02,
                 f'{val:.1f} ms', ha='center', va='bottom', fontweight='bold', fontsize=10)

plt.suptitle('Perbandingan ExtraTrees vs IndoBERT', fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(DOCS_DIR / '05f_model_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

## 13. Test pada 3 Demo Input SIPEDULI

In [ ]:
def predict_indobert(teks):
    indobert_model.eval()
    enc = tokenizer(
        teks, max_length=MAX_LENGTH, padding='max_length',
        truncation=True, return_tensors='pt'
    )
    with torch.no_grad():
        logit = indobert_model(
            enc['input_ids'].to(DEVICE),
            enc['attention_mask'].to(DEVICE)
        ).item()
    # Denormalize: sigmoid(logit) → [0,1] → skala asli [SCORE_MIN, SCORE_MAX]
    skor = torch.sigmoid(torch.tensor(logit)).item() * SCORE_RANGE + SCORE_MIN
    skor = float(np.clip(skor, 0, 100))
    kat  = 'Tinggi' if skor >= THR_TINGGI else 'Sedang' if skor >= THR_SEDANG else 'Rendah'
    return skor, kat

demo_tests = [
    ('Tinggi', 'Perampokan Bersenjata',
     'Pak polisi mohon segera ke sini! Tadi malam ada dua orang bersenjata masuk paksa ke warung makan saya, mereka bawa parang dan todong saya. Ada korban luka parah di kepala, kondisinya kritis. Kami butuh bantuan segera.'),
    ('Sedang', 'Pencurian Kendaraan',
     'Selamat pagi pak, saya mau lapor pencurian motor. Motor saya Honda Scoopy plat D 5521 AB raib dari garasi tadi subuh. Gembok dipaksa rusak, ada rekaman CCTV tapi kurang jelas. Mohon bantuan pelacakan.'),
    ('Rendah', 'Penipuan Online',
     'Selamat siang pak, saya mau lapor penipuan online. Saya beli baju lewat marketplace seharga Rp 150 ribu, barang tidak dikirim dan penjual tidak merespons. Saya punya bukti transaksi lengkap.'),
    ('Rendah', 'SAFE_CONTEXT — beli pisau di toko',
     'tadi beli pisau dapur di toko peralatan masak, harganya murah dan kualitasnya bagus'),
]

print('=' * 72)
print('  PERBANDINGAN PREDIKSI PADA DEMO INPUT SIPEDULI')
print('=' * 72)
print(f'{"Kasus":<28} {"Target":<8} {"ExtraTrees (raw)":<20} {"IndoBERT"}')
print('-' * 72)

for target, nama, teks in demo_tests:
    skor_et, kat_et     = predict_extratrees_raw(teks)
    skor_bert, kat_bert = predict_indobert(teks)

    ok_et   = 'OK' if kat_et   == target else 'SALAH'
    ok_bert = 'OK' if kat_bert == target else 'SALAH'

    print(f'{nama:<28} {target:<8} '
          f'{kat_et} ({skor_et:.1f}) [{ok_et}]   '
          f'{kat_bert} ({skor_bert:.1f}) [{ok_bert}]')

---
# BAGIAN 4 — Kesimpulan

In [ ]:
print('=' * 65)
print('  KESIMPULAN PERBANDINGAN MODEL')
print('=' * 65)
print()
print('[METRIK EVALUASI]')
print(f'  ExtraTrees  — R²: {ET_R2:.4f}  RMSE: {ET_RMSE:.4f}  MAE: {ET_MAE:.4f}')
print(f'  IndoBERT    — R²: {INDOBERT_R2:.4f}  RMSE: {INDOBERT_RMSE:.4f}  MAE: {INDOBERT_MAE:.4f}')
print()
print('[KEUNGGULAN IndoBERT]')
print('  + Memahami konteks kalimat (beli pisau di toko != orang bawa pisau)')
print('  + Tidak butuh Sastrawi stemming')
print('  + Pre-trained pada corpus Indonesia yang besar')
print('  + Lebih sedikit/tanpa post-processing rules')
print()
print('[KEUNGGULAN ExtraTrees]')
print('  + Inferensi ~100x lebih cepat')
print('  + Ukuran model ~50x lebih kecil')
print('  + RAM ~10x lebih hemat')
print('  + SHAP TreeExplainer kompatibel')
print('  + Bisa deploy di Railway free tier (512 MB)')
print('  + Training jauh lebih cepat (20 menit vs berjam-jam)')
print()
print('[KEPUTUSAN DEPLOYMENT]')
print('  PRODUCTION (Railway): ExtraTrees + SHAP')
print('  Alasan: constraint memory Railway tidak memungkinkan load IndoBERT')
print('          (~450 MB model + tokenizer + runtime = OOM di free tier)')
print()
print('  ALTERNATIF MASA DEPAN:')
print('  - Kuantisasi IndoBERT (INT8) → ukuran ~110 MB')
print('  - Deploy IndoBERT di layanan terpisah (HuggingFace Spaces)')
print('  - Fine-tune model IndoBERT yang lebih kecil (distilbert-indonesian)')